#Extracting Feature Vectors from Specific ViT Layers

In [ ]:
import torch
from torchvision.models import vit_b_16, ViT_B_16_Weights

In [ ]:
def extract_feature_vectors(image_tensor):
  #Load pretrained ViT model
  weights = ViT_B_16_Weights.DEFAULT
  model = vit_b_16(weights=weights)
  model.eval()

  #Register a forward hook to capture feature vectors
  feature_vectors = []

  def hook_fn(module,input,output):
    #Capture the output of the second transformer layer
    feature_vectors.append(output)

  #Register the hook on the second transformer block
  second_transformer_block = model.encoder.layers[1]
  hook_handle = second_transformer_block.register_forward_hook(hook_fn)

  #Adding batch dimension
  batched_image = image_tensor.unsqueeze(0)

  #Pass the image through the model
  with torch.no_grad():
    model(batched_image)

  #Remove the hook
  hook_handle.remove()

  #Get features vectors from second layer
  #The first element (CLS token) often contains special meaning
  #Shape: (batch_size=1, num_patches+1, hidden_dimension)
  #where num_patches+1 includes the CLS token
  features = feature_vectors[0]

  return features



In [ ]:
image_tensor = torch.rand(3,224,224)
features = extract_feature_vectors(image_tensor)
print(features.shape)

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth
100%|██████████| 330M/330M [00:05<00:00, 65.8MB/s]


torch.Size([1, 197, 768])
